# TVG 자료 확인
전체 세포 사진과 원본 설명·위치 주석을 확인합니다. 질환 학습 노트북이 아닙니다.


In [ ]:
from pathlib import Path
import pandas as pd
import json

BASE = Path(r"C:\chromosome_ai\tvg_stage2")
assert (BASE / "case_catalog.csv").is_file(), "압축을 푼 폴더 위치를 확인해줘."
catalog = pd.read_csv(BASE / "case_catalog.csv", keep_default_na=False)
paired = catalog[catalog["acquisition_status"] == "paired"].copy()
print("원본 이상 설명 행:", len(catalog))
print("사진·주석 확보:", len(paired))
display(paired.groupby("source_category").size().rename("사진 수").to_frame())
display(catalog.head())


In [ ]:
missing = []
for _, row in paired.iterrows():
    for column in ["image_path", "annotation_path"]:
        if not (BASE / row[column]).is_file():
            missing.append(row[column])
print("로컬 누락 파일:", len(missing))
assert not missing, missing[:5]
report = json.loads((BASE / "audit_report.json").read_text(encoding="utf-8"))
print("CSV 총수 / XML 객체 수 불일치:", len(report["csv_total_xml_object_count_mismatches"]))
print("질환명 정답 제공:", report["disease_ground_truth_available"])
print("환자 그룹 정보 제공:", report["patient_grouping_available"])


## 원본 주석으로 21번 위치 확인
아래 상자는 데이터 제공자의 정답 주석입니다. AI 검출 결과나 질환 확진 결과가 아닙니다.


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import xml.etree.ElementTree as ET

choices = paired[
    (paired["source_category"] == "number_abnormalities") &
    (paired["source_affected_chromosome_description"].astype(str) == "21") &
    (paired["source_total_chromosomes"].astype(str) == "47")
]
row = choices.iloc[0]
annotation = ET.parse(BASE / row["annotation_path"])
with Image.open(BASE / row["image_path"]) as source:
    image = source.convert("RGB")
fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(image)
selected = 0
for obj in annotation.findall("object"):
    if obj.findtext("name") != "G21":
        continue
    x1, y1, x2, y2 = [float(obj.findtext("bndbox/" + key)) for key in ["xmin", "ymin", "xmax", "ymax"]]
    ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="#00b894", linewidth=2))
    ax.text(x1, max(0, y1-4), "chr21 annotation", color="#006b52", fontsize=9)
    selected += 1
ax.set_title(str(row["filename"]) + " | Source annotations")
ax.axis("off")
plt.show()
print("원본 CSV의 총수:", row["source_total_chromosomes"])
print("원본 CSV의 이상 관련 번호:", row["source_affected_chromosome_description"])
print("XML에 표시된 21번 객체 수:", selected)
print("질환명 정답은 제공되지 않았어. 현재 화면은 원본 주석 확인용이야.")
